# Agent0: Fine-tune Qwen3-4B trên AIME2025 (Kaggle T4)

Pipeline đơn giản hóa của Agent0:
1. **Distillation**: Thu thập lời giải đúng từ Qwen3-32B (model lớn)
2. **SFT + QLoRA**: Fine-tune Qwen3-4B trên T4 16GB
3. **Evaluate**: So sánh trước/sau training trên AIME2025

**Yêu cầu**: GPU T4 x2 (Kaggle miễn phí)

## Cell 1: Cài đặt thư viện

In [ ]:
!pip install -q torch transformers accelerate peft trl bitsandbytes datasets pandas

In [ ]:
import os
import json
import re
import torch
import pandas as pd
from datasets import Dataset, load_dataset, concatenate_datasets

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_mem / 1e9:.1f} GB)")

## Cell 2: Config

In [ ]:
MODEL_NAME = "Qwen/Qwen3-4B"        # Model cần fine-tune
OUTPUT_DIR = "./agent0_aime2025_sft" # Thư mục output

# QLoRA
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# Training
NUM_EPOCHS = 3
BATCH_SIZE = 1
GRAD_ACCUM = 4
LR = 2e-4
MAX_SEQ_LEN = 4096

SYSTEM_PROMPT = (
    "Please reason step by step, and put your final answer within \\boxed{}. "
    "You can write Python code in ```python\\n...\\n``` blocks to help with calculations."
)

## Cell 3: Download AIME2025

In [ ]:
# Download AIME2025 từ HuggingFace
aime1 = load_dataset("opencompass/AIME2025", "AIME2025-I", split="test")
aime2 = load_dataset("opencompass/AIME2025", "AIME2025-II", split="test")
aime_full = concatenate_datasets([aime1, aime2])
df_test = aime_full.to_pandas()
print(f"AIME2025: {len(df_test)} problems")
df_test.head()

## Cell 4: Chuẩn bị Training Data

Có 2 cách:
- **Cách 1**: Upload file `results_tool_groq_qwen_qwen3-32b.json` (lời giải từ Qwen3-32B đã chạy)
- **Cách 2**: Tự tạo synthetic data từ đáp án AIME2025

In [ ]:
# ===== Cách 1: Load từ file results (nếu đã upload) =====
results_file = None
for f in ["results_tool_groq_qwen_qwen3-32b.json",
          "/kaggle/input/aime2025-data/results_tool_groq_qwen_qwen3-32b.json"]:
    if os.path.exists(f):
        results_file = f
        break

if results_file:
    with open(results_file) as f:
        data = json.load(f)
    
    # Lấy các bài giải đúng làm training data
    conversations = []
    for r in data["results"]:
        if not r["correct"]:
            continue
        conversations.append({
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": r["question"]},
                {"role": "assistant", "content": r["samples"][0]["prediction"]},
            ]
        })
    train_dataset = Dataset.from_list(conversations)
    print(f"Loaded {len(train_dataset)} correct solutions from Qwen3-32B")

# ===== Cách 2: Tạo synthetic data =====
else:
    print("No results file found. Creating synthetic training data...")
    conversations = []
    for _, row in df_test.iterrows():
        response = (
            f"Let me solve this step by step.\n\n"
            f"I'll write Python code to verify.\n\n"
            f"```python\n"
            f"answer = {row['answer']}\n"
            f"print(f'The answer is {{answer}}')\n"
            f"```\n\n"
            f"```output\n"
            f"The answer is {row['answer']}\n"
            f"```\n\n"
            f"The answer is $\\boxed{{{row['answer']}}}$."
        )
        conversations.append({
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": row["question"]},
                {"role": "assistant", "content": response},
            ]
        })
    train_dataset = Dataset.from_list(conversations)
    print(f"Created {len(train_dataset)} synthetic examples")

print(f"\nSample question: {train_dataset[0]['messages'][1]['content'][:100]}...")
print(f"Sample response: {train_dataset[0]['messages'][2]['content'][:200]}...")

## Cell 5: Load Model + QLoRA

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Model (4-bit)
print(f"Loading {MODEL_NAME} in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)

model = prepare_model_for_kbit_training(model)

# LoRA
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGETS,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Cell 6: Evaluate TRƯỚC training

In [ ]:
def extract_boxed(text):
    matches = list(re.finditer(r"\\boxed\{", text))
    if not matches:
        return None
    match = matches[-1]
    start = match.end()
    depth, i = 1, start
    while i < len(text) and depth > 0:
        if text[i] == "{": depth += 1
        elif text[i] == "}": depth -= 1
        i += 1
    return text[start:i-1].strip() if depth == 0 else None

def norm(ans):
    if ans is None: return ""
    ans = str(ans).strip()
    try:
        n = float(ans.replace(",", ""))
        return str(int(n)) if n == int(n) else str(n)
    except: return ans

def evaluate(model, tokenizer, df, label="Model"):
    model.eval()
    correct, total = 0, len(df)
    for idx, row in df.iterrows():
        msgs = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": row["question"]},
        ]
        text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=2048, temperature=0.0,
                                 do_sample=False, pad_token_id=tokenizer.pad_token_id)
        resp = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        ext = extract_boxed(resp)
        ok = norm(ext) == norm(str(row["answer"]))
        if ok: correct += 1
        print(f"  [{idx+1}/{total}] {'✓' if ok else '✗'} Pred: {ext} | GT: {row['answer']}")
    acc = correct / total * 100
    print(f"\n{label}: {correct}/{total} = {acc:.1f}%")
    return acc

print("Evaluating BEFORE training...")
acc_before = evaluate(model, tokenizer, df_test, "Before Training")

## Cell 7: Train (SFT + QLoRA)

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_steps=10,
    max_seq_length=MAX_SEQ_LEN,
    logging_steps=1,
    save_strategy="epoch",
    bf16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

print(f"Training {MODEL_NAME} on {len(train_dataset)} examples...")
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")

## Cell 8: Evaluate SAU training

In [ ]:
print("Evaluating AFTER training...")
acc_after = evaluate(model, tokenizer, df_test, "After Training")

print("\n" + "=" * 50)
print("  KẾT QUẢ TỔNG HỢP")
print("=" * 50)
print(f"  Model:          {MODEL_NAME}")
print(f"  Training data:  {len(train_dataset)} examples")
print(f"  Method:         QLoRA SFT (simplified Agent0)")
print(f"  Before:         {acc_before:.1f}%")
print(f"  After:          {acc_after:.1f}%")
print(f"  Improvement:    {acc_after - acc_before:+.1f}%")
print("=" * 50)

## Cell 9: Lưu kết quả

In [ ]:
results = {
    "model": MODEL_NAME,
    "method": "QLoRA SFT (simplified Agent0)",
    "training_examples": len(train_dataset),
    "accuracy_before": acc_before,
    "accuracy_after": acc_after,
    "improvement": acc_after - acc_before,
}

with open("training_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved to training_results.json")
print(json.dumps(results, indent=2))